# EDA: DualSeq & CADSequence Representations

This notebook loads the configuration from `configs/ft_eightbitbin_fuse_mod.yaml`, extracts a sample `DualSeq` with sequence length `< 8`, prints its 4 representation styles and corresponding output tensors, defines a converter function between `DualSeq` and `CADSequence`, and verifies 1-to-1 convertibility.

In [ ]:
import os
import sys
import random
import yaml
import torch
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer

from utils.pipeline.config import Config
from utils.data_utils.ref_loader import RefLoader
from utils.data_utils.dualseq_dataset import DualSeqDataset, _command_tokenizer
from utils.representations.dual_seq.dual_seq import DualSeq, DualSeqMetadata
from utils.representations.dual_seq.schema import get_dualseq_schema, DEFAULT_COMMANDS
from utils.representations.converter import (
    basic_to_one_head_tokenized,
    basic_to_eight_bit_binarized,
    eight_bit_binarized_to_basic
)
from utils.representations.CadSeqProc.cad_sequence import CADSequence

In [ ]:
# 1. Load configuration from configs/ft_eightbitbin_fuse_mod.yaml
config_path = "configs/ft_eightbitbin_fuse_mod.yaml"
cfg = Config.from_yaml(config_path)
print(f"Loaded config: {cfg.run_name}")

In [ ]:
# 2. Load dataset and pick 1 random sample with sequence length < 8
loader = RefLoader(
    cfg.data.data_folder,
    csv_path=cfg.data.metadata_csv,
    source_data_type=cfg.data.source_data_type,
    max_samples=200
)
df_loaded = loader.load()
dual_seqs = DualSeq.from_text2cad_df(df_loaded)

short_seqs = [ds for ds in dual_seqs if len(ds) < 8]
print(f"Total samples with seq_len < 8: {len(short_seqs)}")

random.seed(cfg.random_seed)
sample_ds = random.choice(short_seqs)
print(f"Picked sample UID: {sample_ds.uid} (seq_len: {len(sample_ds)})")

In [ ]:
# 3. Print representations of DualSeq (cmds, float args, serialized args, 8-bit binned args)
schema = get_dualseq_schema()

# Fit metadata for 8-bit binned args
metadata = DualSeqMetadata()
metadata.fit(dual_seqs)

cmds_repr = sample_ds.cmds
float_args_repr = sample_ds.args
serialized_args_repr = basic_to_one_head_tokenized(sample_ds.cmds, sample_ds.args, schema)
binned_args_repr = basic_to_eight_bit_binarized(sample_ds.cmds, sample_ds.args, metadata)

print("=== DUALSEQ REPRESENTATIONS ===")
print("\n1. Commands (cmd):")
print(cmds_repr)

print("\n2. Float Args:")
print(float_args_repr)

print("\n3. Serialized Arg Tokens:")
print(serialized_args_repr)

print("\n4. 8-Bit Binned Args:")
print(binned_args_repr)

In [ ]:
# 4. Print example of output tensor for the 4 things mentioned above
tokenizer = AutoTokenizer.from_pretrained(cfg.tokenizer.model_name)

# 1) Command Tensor
cmd_tokens = _command_tokenizer(sample_ds.cmds, schema)
cmd_tensor = torch.tensor(cmd_tokens, dtype=torch.long)

# 2) Float Args Tensor
ds_float = DualSeqDataset([sample_ds], tokenizer, description_level="expert", out_type="FloatArgs", metadata=metadata)
_, _, float_args_tensor = ds_float[0]

# 3) Serialized Args Tensor
ds_tokenized = DualSeqDataset([sample_ds], tokenizer, description_level="expert", out_type="TokenizedOneSequenceArgs", metadata=metadata)
_, _, serialized_tokens_list = ds_tokenized[0]
serialized_args_tensor = torch.tensor(serialized_tokens_list, dtype=torch.long)

# 4) 8-Bit Binned Args Tensor
ds_binned = DualSeqDataset([sample_ds], tokenizer, description_level="expert", out_type="EightBitBinarizedArgs", metadata=metadata)
_, _, binned_args_tensor = ds_binned[0]

print("=== OUTPUT TENSORS FOR THE 4 REPRESENTATIONS ===")
print(f"1. Command Tensor (shape {cmd_tensor.shape}):\n{cmd_tensor}\n")
print(f"2. Float Args Tensor (shape {float_args_tensor.shape}):\n{float_args_tensor}\n")
print(f"3. Serialized Args Tensor (shape {serialized_args_tensor.shape}):\n{serialized_args_tensor}\n")
print(f"4. 8-Bit Binned Args Tensor (shape {binned_args_tensor.shape}):\n{binned_args_tensor}")

## CADSequence Converter & 1-to-1 Validation

In [ ]:
def dualseq_to_minimal_json(ds: DualSeq) -> dict:
    """Convert DualSeq cmds and args into text2cad minimal json object format."""
    if hasattr(ds, 'json_object') and ds.json_object and "parts" in ds.json_object and ds.json_object["parts"]:
        return ds.json_object

    parts = {}
    current_part_idx = 0
    i = 0
    n = len(ds.cmds)

    while i < n:
        cmd = ds.cmds[i]
        arg = ds.args[i] if i < len(ds.args) else {}

        if cmd == "COOR":
            current_part_idx += 1
            part_name = f"part_{current_part_idx}"
            euler = [float(arg.get("coor_euax", 0.0)), float(arg.get("coor_euay", 0.0)), float(arg.get("coor_euaz", 0.0))]
            trans = [float(arg.get("coor_tx", 0.0)), float(arg.get("coor_ty", 0.0)), float(arg.get("coor_tz", 0.0))]

            parts[part_name] = {
                "coordinate_system": {
                    "Euler Angles": euler,
                    "Translation Vector": trans
                },
                "sketch": {},
                "extrusion": {}
            }
            i += 1
            face_idx = 0
            loop_idx = 0
            current_face_name = ""
            current_loop_name = ""
            line_idx = 0
            circle_idx = 0
            arc_idx = 0

            while i < n and ds.cmds[i] != "COOR":
                sub_cmd = ds.cmds[i]
                sub_arg = ds.args[i] if i < len(ds.args) else {}

                if sub_cmd == "FACE":
                    face_idx += 1
                    current_face_name = f"face_{face_idx}"
                    parts[part_name]["sketch"][current_face_name] = {}
                    loop_idx = 0
                    i += 1
                elif sub_cmd == "LOOP":
                    loop_idx += 1
                    current_loop_name = f"loop_{loop_idx}"
                    if not current_face_name:
                        face_idx += 1
                        current_face_name = f"face_{face_idx}"
                        parts[part_name]["sketch"][current_face_name] = {}
                    parts[part_name]["sketch"][current_face_name][current_loop_name] = {}
                    line_idx = 0
                    circle_idx = 0
                    arc_idx = 0
                    i += 1
                elif sub_cmd == "LINE":
                    line_idx += 1
                    seg_name = f"line_{line_idx}"
                    line_data = {
                        "Start Point": [float(sub_arg.get("line_sx", 0.0)), float(sub_arg.get("line_sy", 0.0))],
                        "End Point": [float(sub_arg.get("line_ex", 0.0)), float(sub_arg.get("line_ey", 0.0))]
                    }
                    parts[part_name]["sketch"][current_face_name][current_loop_name][seg_name] = line_data
                    i += 1
                elif sub_cmd == "CIRCLE":
                    circle_idx += 1
                    seg_name = f"circle_{circle_idx}"
                    circle_data = {
                        "Center": [float(sub_arg.get("circle_cx", 0.0)), float(sub_arg.get("circle_cy", 0.0))],
                        "Radius": float(sub_arg.get("circle_r", 0.0))
                    }
                    parts[part_name]["sketch"][current_face_name][current_loop_name][seg_name] = circle_data
                    i += 1
                elif sub_cmd == "ARC":
                    arc_idx += 1
                    seg_name = f"arc_{arc_idx}"
                    arc_data = {
                        "Start Point": [float(sub_arg.get("arc_sx", 0.0)), float(sub_arg.get("arc_sy", 0.0))],
                        "Mid Point": [float(sub_arg.get("arc_mx", 0.0)), float(sub_arg.get("arc_my", 0.0))],
                        "End Point": [float(sub_arg.get("arc_ex", 0.0)), float(sub_arg.get("arc_ey", 0.0))]
                    }
                    parts[part_name]["sketch"][current_face_name][current_loop_name][seg_name] = arc_data
                    i += 1
                elif sub_cmd.startswith("EXTRUDE"):
                    if sub_cmd == "EXTRUDE_NEW":
                        op = "NewBodyFeatureOperation"
                        dtn = float(sub_arg.get("extrude_new_dtn", 0.0))
                        don = float(sub_arg.get("extrude_new_don", 0.0))
                        scale = float(sub_arg.get("extrude_new_scale", 1.0))
                    elif sub_cmd == "EXTRUDE_JOIN":
                        op = "JoinFeatureOperation"
                        dtn = float(sub_arg.get("extrude_join_dtn", 0.0))
                        don = float(sub_arg.get("extrude_join_don", 0.0))
                        scale = float(sub_arg.get("extrude_join_scale", 1.0))
                    elif sub_cmd == "EXTRUDE_CUT":
                        op = "CutFeatureOperation"
                        dtn = float(sub_arg.get("extrude_cut_dtn", 0.0))
                        don = float(sub_arg.get("extrude_cut_don", 0.0))
                        scale = float(sub_arg.get("extrude_cut_scale", 1.0))
                    else:
                        op = "IntersectFeatureOperation"
                        dtn = float(sub_arg.get("extrude_intersect_dtn", 0.0))
                        don = float(sub_arg.get("extrude_intersect_don", 0.0))
                        scale = float(sub_arg.get("extrude_intersect_scale", 1.0))

                    parts[part_name]["extrusion"] = {
                        "operation": op,
                        "extrude_depth_towards_normal": dtn,
                        "extrude_depth_opposite_normal": don,
                        "sketch_scale": scale
                    }
                    i += 1
                    break
                else:
                    i += 1
        else:
            i += 1

    return {"parts": parts}

def dualseq_to_cadseq(ds: DualSeq) -> CADSequence:
    """Convert a DualSeq instance to a CADSequence instance."""
    json_obj = dualseq_to_minimal_json(ds)
    return CADSequence.from_minimal_json(json_obj)

def cadseq_to_dualseq(cad_seq: CADSequence, uid: str = "converted") -> DualSeq:
    """Convert a CADSequence instance back to a DualSeq instance."""
    json_obj = cad_seq._json()
    return DualSeq(json_object=json_obj, uid=uid, format="text2cad")

In [ ]:
# Print example of converted CADSequence for the picked sample
cad_seq_sample = dualseq_to_cadseq(sample_ds)

print("=== CONVERTED CADSEQUENCE EXAMPLE ===")
print(cad_seq_sample)

In [ ]:
# Verify 1-to-1 convertibility between DualSeq and CADSequence
reconstructed_ds = cadseq_to_dualseq(cad_seq_sample, uid=sample_ds.uid)

cmds_match = (sample_ds.cmds == reconstructed_ds.cmds)

args_match = True
for a1, a2 in zip(sample_ds.args, reconstructed_ds.args):
    if set(a1.keys()) != set(a2.keys()):
        args_match = False
        break
    for k in a1:
        if abs(a1[k] - a2[k]) > 1e-5:
            args_match = False
            break

print("=== 1-TO-1 CONVERSION VALIDATION ===")
print(f"Original Cmds:      {sample_ds.cmds}")
print(f"Reconstructed Cmds: {reconstructed_ds.cmds}")
print(f"Commands Match:     {cmds_match}")
print(f"Args Match:         {args_match}")

is_valid_convertible = cmds_match and args_match
print(f"Overall 1-to-1 Convertible & Valid: {is_valid_convertible}")
assert is_valid_convertible, "Conversion between DualSeq and CADSequence is not 1-to-1!"